# Cons in this data preparation
1. We are removing the URL, which is essential for ref
2. We haven't deep dive into how the html elements are removed

In [95]:
import pandas as pd
import re
from html import unescape
from datetime import datetime, timezone

In [96]:
df_questions = pd.read_csv('data/Questions.csv', encoding='latin-1')
df_answers = pd.read_csv('data/Answers.csv', encoding='latin-1')

In [97]:
df_questions = df_questions[["Id", "CreationDate", "Title", "Body"]].rename({"Id": "QuestionId", "Body": "QuestionBody", "CreationDate": "QuestionCreationDate"}, axis=1)
df_answers = df_answers[["Id", "CreationDate", "ParentId", "Body"]].rename({"Id": "AnswerId", "Body": "AnswerBody", "CreationDate": "AnswerCreationDate"}, axis=1)

In [98]:
df_stack_data = df_questions.merge(df_answers, how='left', left_on='QuestionId', right_on='ParentId')

In [99]:
df_stack_data.drop(["ParentId"], axis=1, inplace=True)

In [100]:
def remove_html_tags(text):
    if pd.isna(text):
        return text
    text = unescape(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_stack_data["AnswerBody"] = df_stack_data["AnswerBody"].apply(remove_html_tags)
df_stack_data["QuestionBody"] = df_stack_data["QuestionBody"].apply(remove_html_tags)

In [101]:
# Limiting data to questions created after 2014-01-01
# Reducing data from 2176163 to 952556
df_stack_data["QuestionCreationDate"] = pd.to_datetime(df_stack_data["QuestionCreationDate"], errors='coerce')
df_stack_data = df_stack_data[df_stack_data["QuestionCreationDate"] >= datetime(2014, 1, 1, tzinfo= timezone.utc)]

In [ ]:
df_stack_data[["QuestionId", "Title", "QuestionBody", "AnswerBody"]].groupby(["QuestionId", "Title", "QuestionBody"])[["AnswerBody"]]

TypeError: 'bool' object is not callable

In [72]:
df_stack_data = pd.DataFrame(df_stack_data[["QuestionId", "Title", "QuestionBody", "AnswerBody"]].groupby(["QuestionId", "Title", "QuestionBody"])["AnswerBody"].apply(list).reset_index())

In [90]:
df_stack_data.columns

Index(['QuestionId', 'Title', 'QuestionBody', 'AnswerBody'], dtype='str')

In [94]:
df_stack_data.dropna(axis=0)

,QuestionId,Title,QuestionBody,AnswerBody
0,20864430,PHP mailing address preg_match() not working,I am trying to make a regular expression to ma...,[Try this RE: /[1-9][a-z] [1-9][0-9] [a-z.]+ [...
1,20864450,generating json for google charts - adding nulls,I need to generate a null value in json if a r...,[I was close. This works... foreach ( $results...
2,20864470,Polymorphism and inheritance in Avro schemas,Is it possible to write an Avro schema/IDL tha...,[I decided to use the ReflectData API to gener...
3,20864520,Microsoft Word Highlighting Text White,"For some reason, whenever I create a Text Box ...",[Right-click the text box that you want to mak...
4,20864570,How do you write a pl sql script that iterates...,I have a database that looks like : http://sql...,[I just free typed this cause I'm away from my...
...,...,...,...,...
658754,40143210,URL routing in PHP (MVC),I am building a custom MVC project and I have ...,[nan]
658755,40143300,Bigquery.Jobs.Insert - Resumable Upload?,The API docs show that you should be able to m...,[nan]
658756,40143340,Obfuscating code in android studio,Under minifyEnabled I changed from false to tr...,[nan]
658757,40143360,How to fire function after v-model change?,I have input which I use to filter my array of...,[nan]


In [ ]:
def format_stack_data(row):
    answers = " | ".join(row["AnswerBody"]) if isinstance(row["AnswerBody"], list) else ""
    return (
        f"Question title: {row['Title']}\n"
        f"Question body: {row['QuestionBody']}\n"
        f"Answers: {answers}"
    )

formatted_rows = df_stack_data.apply(format_stack_data, axis=1)
formatted_rows.head()

TypeError: sequence item 0: expected str instance, float found

In [74]:
df_stack_data.to_csv('data/StackOverflow_Cleaned.csv', index=False, encoding='utf-8')